# 06 — Principal Component Analysis

**Objective:** Plan PCA experiments and interpretation.  
**Owner:** Ilias El Hamri  
**Sprint:** 01  

> Leakage warning: fit scaling and PCA on training folds only.

## Project Setup

This cell locates the project root and loads the shared configuration.

- The project root is found by walking up from the current working directory until a folder containing `configs/config.yaml` is found. This ensures the notebook works regardless of where Jupyter is launched from.
- The project root is inserted into `sys.path` so that `src.*` imports resolve correctly.
- `load_config()` reads `configs/config.yaml` and returns a dictionary that contains shared settings such as `random_state`, output paths, and metric names.

In [ ]:
from pathlib import Path

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs" / "config.yaml").is_file())
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import load_config

config = load_config()
print(config["project"]["name"])
print(config["project"]["random_state"])

## Planned work

Later: evaluate explained variance and downstream performance using fold-local transformations and the shared validation protocol.

## Data Loading

This cell loads the Santander Customer Transaction dataset with the shared memory-optimized loader, creates the official split, and verifies both partition fingerprints.

- `optimize_memory=True` converts numeric features from `float64` to `float32`, reducing memory usage by roughly half while preserving all structural invariants.
- The function returns three objects:
  - `X` — a DataFrame of 200 anonymised feature columns.
  - `y` — a binary target Series (`"0"` / `"1"`).
  - `metadata` — a dictionary with dataset description and provenance information.

Only `X_train` and `y_train` remain available after fingerprint verification. The reserved partition is explicitly deleted.

In [ ]:
from src.data import load_dataset
from src.validation import create_train_test_split, split_fingerprint

EXPECTED_TRAIN_FINGERPRINT = "61c403ec521d15ab9d6316606eba5acdfc22381cb764b6da76a65041ec11f477"
EXPECTED_RESERVED_FINGERPRINT = "bf7d43e492967dad1358676e9bf8355a910823077b59118014db12af3e26f586"

X, y, metadata = load_dataset(optimize_memory=True)
X_train, X_reserved, y_train, y_reserved = create_train_test_split(X, y)
assert split_fingerprint(X_train.index) == EXPECTED_TRAIN_FINGERPRINT
assert split_fingerprint(X_reserved.index) == EXPECTED_RESERVED_FINGERPRINT
del X_reserved, y_reserved, X, y

## Pipeline Construction

This cell obtains the production PCA pipeline from `src.feature_selection`; it does not duplicate its implementation.

### Why scale before PCA?

- PCA is based on variance decomposition. Without scaling, features with larger numeric ranges dominate the principal components regardless of their predictive importance.
- `StandardScaler` normalises all 200 features to zero mean and unit variance before PCA is applied.

### PCA configuration

- `n_components=0.95` instructs PCA to automatically select the **minimum number of components** needed to preserve 95% of the total explained variance.
- This avoids guessing a fixed number of components and adapts to the data in each fold.
- `random_state` ensures reproducibility across runs.

### Final classifier

- A standard L2-regularised Logistic Regression trains on the compressed PCA components, not the original 200 features.

### Why a Pipeline?

- Bundling all three steps — scaling → PCA → classification — inside a `Pipeline` is mandatory.
- The registered experiment fitted the entire pipeline independently inside each cross-validation training fold, so `StandardScaler` and `PCA` were never exposed to validation or reserved data.

In [ ]:
from src.feature_selection import create_pca_pipeline

pipeline = create_pca_pipeline()
pipeline

## Registered Cross-Validation Results

This cell reads the immutable artifacts produced earlier by the registered experiment. It performs no fitting and writes no result.

- The registered run applied 5-fold stratified cross-validation on **training data only**.
- It loads two existing objects:
  - `fold_results` — a DataFrame with per-fold metrics (ROC-AUC, Average Precision, F1, etc.).
  - `summary` — a dictionary with aggregate scores, estimator parameters, and experiment metadata.

The final test partition is reserved and was not used for feature selection, PCA selection, hyperparameter selection, or model comparison.

In [ ]:
import json
import pandas as pd

fold_results = pd.read_csv(project_root / "reports/experiments/M03-PCA-001_fold_results.csv")
summary = json.loads((project_root / "reports/experiments/M03-PCA-001_summary.json").read_text(encoding="utf-8"))
display(fold_results)
print(f"Primary Metric ({summary['primary_metric']}): {summary['primary_score_mean']:.4f} +/- {summary['primary_score_std']:.4f}")